## 0. Install dependencies (fresh Colab GPU runtime)

In [ ]:
%pip install -U unsloth pandas datasets scikit-learn nbformat

## 1. Configure baseline and files

In [ ]:
from pathlib import Path
import os, re, json, hashlib, random, time, platform, sys, importlib.metadata
import pandas as pd

SYSTEM_KEY = "zookeeper"   # bgl / apache / linux / zookeeper
APPROACH = "stratified" # or random
SUMMARY_TYPE = "abstractive"  # extractive / abstractive / hybrid
# Set the path to your teacher-labeled TEST CSV (the only dataset file required).
TEST_FILE = Path("/...")

def resolve_csv_path(path):
    alt = path.with_name(path.stem + "(1)" + path.suffix)
    return alt if not path.exists() and alt.exists() else path
TEST_FILE = resolve_csv_path(TEST_FILE)
EVAL_FILE = TEST_FILE
INFERENCE_SPLIT = "test"

MODEL_NAME = "unsloth/Qwen3-8B-bnb-4bit"
LOAD_IN_4BIT = True

SEED = 3407
MODES = ["zero_shot"] # | few-shot incase need to check few-shot
FEW_SHOT_K = 2  # incase need to check few-shot
# Demonstrations are FIXED in Section 4, not sampled at runtime.

MAX_SEGMENTS = None # None = all; use e.g. 20 for smoke test
INFERENCE_RESUME = False
RESUME_RETRY_FAILED = False  # False: skip attempted failed rows; True: retry them on resume

OUTPUT_ROOT = Path("/content/logsumm_baselines")
# Qwen3 should be consistently in non-thinking mode across experiments.
USE_CHAT_TEMPLATE = False

# Reduced output budgets for faster inference. Increase if legitimate long outputs hit the cap.
# The first number is model context capacity; the second is the generation budget.
SUMMARY_TOKEN_CONFIG = {
    "extractive": (4096, 600),
    "abstractive": (4096, 600),
    "hybrid": (6144, 1500),
}

# Show each finished model response immediately. Longer output is abbreviated only in the console,
# while the complete raw response continues to be saved in the CSV.
PRINT_EACH_RESPONSE = True
PRINT_MAX_CHARS = 5000  # None prints the entire raw response; smaller values reduce notebook output.
CHECKPOINT_EVERY = 10

MAX_SEQ_LENGTH, MAX_NEW_TOKENS = SUMMARY_TOKEN_CONFIG[SUMMARY_TYPE]

INPUT_COLUMN = "Content_without_labels"
TARGET_COLUMN = "GT"
RUN_SUMMARY_TYPE = SUMMARY_TYPE
LABEL_TAG = re.compile(r"\[label=[^\]\r\n]*\]", re.I)
ID_TAG = re.compile(r"\[id=([^\]\r\n]+)\]")
for x, choices in [(SYSTEM_KEY, {'bgl','apache','linux','zookeeper'}),
                   (SUMMARY_TYPE, {'extractive','abstractive','hybrid'}),
                   (INFERENCE_SPLIT, {'test'})]:
    if x not in choices: raise ValueError(f"Invalid setting: {x}")
if USE_CHAT_TEMPLATE:
    raise ValueError("Chat templates change V2.4 prompt parity. Keep USE_CHAT_TEMPLATE=False until both experiments are updated consistently.")
if not set(MODES) <= {'zero_shot','few_shot'} or not MODES:
    raise ValueError("Select at least one valid mode")
if FEW_SHOT_K < 1 and 'few_shot' in MODES:
    raise ValueError("Few-shot requires K >= 1")

def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False, default=str).encode()).hexdigest()

def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1048576), b''): h.update(block)
    return h.hexdigest()

def save_json(value,path):
    p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)
    temp=p.with_suffix(p.suffix+'.tmp'); temp.write_text(json.dumps(value,ensure_ascii=False,indent=2,default=str),encoding='utf-8'); temp.replace(p)

def save_csv(frame,path):
    p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)
    temp=p.with_suffix(p.suffix+'.tmp'); frame.to_csv(temp,index=False,encoding='utf-8-sig'); temp.replace(p)


In [ ]:
# Google Colab downloads are triggered at the END of inference only.
# On local Jupyter, files are retained on disk and their paths are printed.
AUTO_DOWNLOAD_PREDICTIONS = True

def auto_download_prediction_csvs(paths):
    paths = [Path(p) for p in paths if Path(p).is_file()]
    if not AUTO_DOWNLOAD_PREDICTIONS or not paths:
        return
    try:
        from google.colab import files as colab_files
    except ImportError:
        print("Not running in Colab; generated CSVs:", [str(p) for p in paths])
        return
    for path in paths:
        print("Downloading inference CSV:", path)
        colab_files.download(str(path))


## 2. Teacher-aligned student instructions and existing output validators

In [ ]:
# Student prompts adapted directly from the teacher's PROMPTS_EXPLANATION.
# Teacher-only event labels are unavailable to the student. All three student
# experiments share these same instructions; few-shot only adds examples.
SHARED_INSTRUCTIONS = """"""

SYSTEM_DESCRIPTIONS = {'bgl': 'IBM BlueGene/L supercomputer',
                       'apache': 'Apache HTTP server. For unknown events, distinguish genuine web-server or application failures from expected request handling and routine operation. Treat an event as a fault only when the logs provide clear evidence of an unsuccessful internal operation, malfunction, or abnormal service behavior. Client-side errors, rejected requests, and diagnostic messages are not faults unless they reveal an underlying system problem. Consider the event context and outcome rather than severity or error keywords alone.',
                       'linux': 'Linux system logs. Look for crashes and uncorrected disk/kernel faults; routine cron and successful logins are background.',
                       'zookeeper': 'Apache ZooKeeper service. Look for failed elections, expired sessions with impact and unrecoverable errors; routine disconnects are background.'}

PROMPTS = {
    'extractive': 'For each event, decide if it reports a real fault from its content and context. Select a fault only when it clearly shows a real failure (e.g., a crash, uncorrected hardware/I/O fault, or unsuccessful operation). Corrected or recovered notices, diagnostics, counters, periodic messages, and uncertain cases are background. Do not infer unsupported failures or outcomes.\nProvide a nonempty, brief explanation of the main selections. If no faults, begin explanation with "Everything looks normal."\ncritical_events: all distinct fault messages, one original representative each, in input order.\ncontext_events: Choose up to 3 informative background events: lifecycle transitions,\nnearby same-component context, or rare changes. Exclude repeated routine activity,\nstatus/counters, diagnostics, and already-corrected notices. Keep input order.\nCopy selected lines EXACTLY; retain [id=...].\nexplanation and lists must agree. Return JSON only:\n{"explanation":"...","critical_events":[],"context_events":[]}',
    'abstractive': 'For each event, decide if it reports a real fault from its content and context. Select a fault only when it clearly shows a real failure (e.g., a crash, uncorrected hardware/I/O fault, or unsuccessful operation). Corrected or recovered notices, diagnostics, counters, periodic messages, and uncertain cases are background. Do not infer unsupported failures or outcomes.\nWrite a brief, nonempty explanation of the main selections. If no faults, begin explanation with "Everything looks normal." Qualify inferred faults.\nWrite ONE short paragraph: \'The fault:\' followed by faults, affected components and\nobserved consequences; count repeats if useful. Then \'For background,\' with only\nqualifying context: Choose up to 3 informative background events: lifecycle transitions,\nnearby same-component context, or rare changes. Exclude repeated routine activity,\nstatus/counters, diagnostics, and already-corrected notices. Keep input order.\nIf no faults, start summary with "Everything looks normal." and optionally add relevant background.\nDo not invent outcomes or copy full lines. Return JSON only:\n{"explanation":"...","summary":"..."}',
    'hybrid': 'For each event, decide if it reports a real fault from its content and context. Select a fault only when it clearly shows a real failure (e.g., a crash, uncorrected hardware/I/O fault, or unsuccessful operation). Corrected or recovered notices, diagnostics, counters, periodic messages, and uncertain cases are background. Do not infer unsupported failures or outcomes.\nProvide a nonempty, brief explanation of the main selections. If no faults, begin explanation with "Everything looks normal." Qualify inferred faults.\ncritical_events: every distinct fault message (one exact original line per message).\ncontext_events: Choose up to 3 informative background events: lifecycle transitions,\nnearby same-component context, or rare changes. Exclude repeated routine activity,\nstatus/counters, diagnostics, and already-corrected notices. Keep input order.\nCopy selected lines exactly, preserving [id=...] and order.\nWhen critical_events is nonempty, analysis must contain:\n- fault_category: hardware, memory, storage_io, network, application, configuration, resource, security, unknown,\n- affected_components: verbatim IDs,\n- recovery_state: recovered, unrecovered, unknown,\n- likely_cause: \'undetermined\' if unsupported,\n- evidence_ids: critical-event IDs only.\nAdd a 1-2 sentence interpretation without invented consequences. If no faults, analysis=null and interpretation="Everything looks normal."; context_events can be nonempty.\nReturn JSON only:\n{"explanation":"...","critical_events":[],"context_events":[],\n"analysis":{"fault_category":"...","affected_components":[],"recovery_state":"...",\n"likely_cause":"...","evidence_ids":[]},"interpretation":"..."}',
}

# Resolve teacher-style system placeholder without changing the template wording.
PROMPTS = {name: template.replace("{SYSTEM_FULL}", SYSTEM_DESCRIPTIONS[SYSTEM_KEY])
           for name, template in PROMPTS.items()}

RUN_INSTRUCTION = (SYSTEM_DESCRIPTIONS[SYSTEM_KEY] + "\n" +
                   SHARED_INSTRUCTIONS.strip() + "\n\n" +
                   PROMPTS[RUN_SUMMARY_TYPE].strip())

def build_prompt(chunk):
    if not isinstance(chunk, str) or not chunk.strip():
        raise ValueError("Empty log chunk")
    if LABEL_TAG.search(chunk) or re.search(r"\bEventTemplate\s*=", chunk, flags=re.I):
        raise ValueError("Teacher-only field found in model input")
    return RUN_INSTRUCTION + "\n\nLog chunk:\n" + chunk + "\n\n### Response:\n"


def whitespace_key(text):
    return re.sub(r"\s+", " ", text).strip()


def source_lines(chunk):
    lines = [x for x in chunk.splitlines() if x.strip()]
    ids = [ID_TAG.findall(x) for x in lines]
    if any(len(x) != 1 for x in ids) or len({x[0] for x in ids}) != len(ids):
        raise ValueError("Each source event requires a unique [id=...] tag")
    return lines


def event_identifier(event, index):
    for key in ("LineId", "line_id", "lineid", "id"):
        if event.get(key) is not None and str(event[key]).strip():
            return str(event[key]).strip()
    # Older or normalized datasets may omit LineId: match the teacher's
    # assigned identifier only through the supplied Content_without_labels.
    return None


def validate_target(value, chunk):
    if not isinstance(value, str):
        raise ValueError("Summary must be JSON text")
    obj = json.loads(value.strip())
    required = ({"explanation", "summary"} if RUN_SUMMARY_TYPE == "abstractive" else
                {"explanation", "critical_events", "context_events"})
    if RUN_SUMMARY_TYPE == "hybrid":
        required |= {"analysis", "interpretation"}
    if not isinstance(obj, dict) or set(obj) != required:
        raise ValueError(f"Expected {sorted(required)}; received {list(obj) if isinstance(obj, dict) else type(obj)}")
    if LABEL_TAG.search(value) or re.search(r"\bevent[_\s-]*template\s*=", value, re.I):
        raise ValueError("Forbidden ground-truth/template field in summary")
    explanation = obj["explanation"]
    if not isinstance(explanation, str) or not explanation.strip():
        raise ValueError("explanation must be nonempty")
    lines = source_lines(chunk)
    if RUN_SUMMARY_TYPE == "abstractive":
        summary = obj["summary"]
        if not isinstance(summary, str) or not summary.strip():
            raise ValueError("abstractive summary must be nonempty")
        if not summary.startswith(("The fault:", "Everything looks normal.")):
            raise ValueError("Unexpected abstractive prefix")
        return json.dumps(obj, ensure_ascii=False, allow_nan=False)
    lookup = {whitespace_key(line): (i, line) for i, line in enumerate(lines)}
    used = set()
    for key in ("critical_events", "context_events"):
        events = obj[key]
        if not isinstance(events, list) or any(not isinstance(x, str) or not x.strip() for x in events):
            raise ValueError(f"Invalid {key}")
        if key == "context_events" and len(events) > 3:
            raise ValueError("At most three context events")
        previous = -1
        for j, event in enumerate(events):
            match = lookup.get(whitespace_key(event))
            if match is None: raise ValueError(f"{key} contains invented or edited event")
            position, canonical = match
            if position in used or position <= previous:
                raise ValueError("Duplicate event or incorrect order")
            previous = position
            used.add(position)
            events[j] = canonical
    if not obj["critical_events"] and "Everything looks normal." not in explanation:
        raise ValueError("No-fault explanation requires standard phrase")
    if RUN_SUMMARY_TYPE == "hybrid":
        analysis = obj["analysis"]
        interp = obj["interpretation"]
        if not isinstance(interp, str) or not interp.strip():
            raise ValueError("interpretation must be nonempty")
        if not obj["critical_events"]:
            if analysis is not None or "Everything looks normal." not in interp:
                raise ValueError("Invalid no-fault hybrid output")
        else:
            fields = {"fault_category", "affected_components", "recovery_state", "likely_cause", "evidence_ids"}
            if not isinstance(analysis, dict) or set(analysis) != fields:
                raise ValueError("Incorrect analysis schema")
            if analysis["fault_category"] not in {"hardware", "memory", "storage_io", "network", "application", "configuration", "resource", "security", "unknown"}:
                raise ValueError("Invalid fault category")
            if analysis["recovery_state"] not in {"recovered", "unrecovered", "unknown"}:
                raise ValueError("Invalid recovery state")
            for field in ("affected_components", "evidence_ids"):
                if not isinstance(analysis[field], list) or any(not isinstance(t, str) for t in analysis[field]):
                    raise ValueError(f"Invalid {field}")
            ids = {ID_TAG.findall(line)[0] for line in obj["critical_events"]}
            if not set(analysis["evidence_ids"]) <= ids:
                raise ValueError("Unsupported evidence ID")
            if not isinstance(analysis["likely_cause"], str):
                raise ValueError("Invalid likely_cause")
    return json.dumps(obj, ensure_ascii=False, allow_nan=False)


def summary_payload(value):
    obj = json.loads(value) if isinstance(value, str) else value
    return obj["summary"] if RUN_SUMMARY_TYPE == "abstractive" else {k: v for k, v in obj.items() if k != "explanation"}


def has_fault_output(value):
    obj = json.loads(value) if isinstance(value, str) else value
    return obj["summary"].startswith("The fault:") if RUN_SUMMARY_TYPE == "abstractive" else bool(obj["critical_events"])


def validate_reference(value, chunk, row):
    """Validate teacher target shape and preservation. Known event labels override inference.
    Unknown-labeled events remain teacher-inferred, not ground-truth anomalies.
    Never use source annotations as student input.
    """
    canonical = validate_target(value, chunk)
    if not row.get("events_json", ""):
        return canonical
    events = json.loads(row["events_json"])
    if not isinstance(events, list): raise ValueError("Invalid events_json")
    known_fault_ids, known_normal_ids = set(), set()
    for i, event in enumerate(events):
        eid = event_identifier(event, i)
        lab = str(event.get("Label", event.get("label", "unknown"))).strip().lower()
        if eid:
            if lab == "-": known_normal_ids.add(eid)
            elif lab not in ("unknown", "", "none", "nan"): known_fault_ids.add(eid)
    obj = json.loads(canonical)
    if RUN_SUMMARY_TYPE != "abstractive":
        critical_ids = {ID_TAG.findall(line)[0] for line in obj["critical_events"]}
        context_ids = {ID_TAG.findall(line)[0] for line in obj["context_events"]}
        if known_fault_ids - critical_ids:
            # Deduplication can keep one representative of an identical message.
            by_id = {event_identifier(e,i):str(e.get("Content", "")) for i,e in enumerate(events)}
            selected_messages = {by_id.get(eid) for eid in critical_ids}
            if any(by_id.get(eid) not in selected_messages for eid in known_fault_ids - critical_ids):
                raise ValueError("Confirmed fault message omitted from critical events")
        if critical_ids & known_normal_ids or context_ids & known_fault_ids:
            raise ValueError("Teacher critical/context conflicts with known labels")
    elif known_fault_ids and not has_fault_output(obj):
        raise ValueError("Confirmed fault omitted from abstractive summary")
    return canonical


## 3. Input preprocessing and teacher reference validation

In [ ]:
def get_student_input(row):
    text = row.get(INPUT_COLUMN, "")
    if isinstance(text, str) and text.strip():
        if LABEL_TAG.search(text):
            raise ValueError(f"{INPUT_COLUMN} contains event ground truth")
        if re.search(r"\bEventTemplate\s*=", text, re.I):
            raise ValueError("EventTemplate metadata in input: regenerate references with the corrected teacher notebook")
        source_lines(text)
        return text
    if row.get("events_json", ""):
        events = json.loads(row["events_json"])
        if not isinstance(events, list) or not events: raise ValueError("Invalid events_json")
        lines = []
        for index, event in enumerate(events):
            if not isinstance(event, dict) or not isinstance(event.get("Content"), str):
                raise ValueError("Each source event requires Content")
            eid = event_identifier(event, index)
            if not eid:
                raise ValueError("events_json lacks stable IDs; supply Content_without_labels")
            metadata = [f"{k}={v}" for k, v in event.items()
                        if k.lower().replace("_", "") not in {"lineid","id","eventid","label","content","eventtemplate"}
                        and v is not None and str(v).strip()]
            lines.append(f"[id={eid}] " + (" ".join(metadata) + " " if metadata else "") + event["Content"])
        text = "\n".join(lines)
    else:
        raise ValueError(f"Need {INPUT_COLUMN} or events_json")
    if LABEL_TAG.search(text):
        raise ValueError("Ground-truth event field remains in student input")
    source_lines(text)
    return text


def load_frame(path, split, require_targets=True):
    frame = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    if frame.empty or "segment_id" not in frame:
        raise ValueError(f"{path}: empty data or missing segment_id")
    if frame.segment_id.duplicated().any() or frame.segment_id.str.strip().eq("").any():
        raise ValueError(f"{path}: nonunique or empty segment_id")
    if "split" in frame and not frame["split"].eq(split).all():
        raise ValueError(f"{path}: incorrect split field")
    if require_targets and TARGET_COLUMN not in frame:
        raise ValueError(f"{path}: missing {TARGET_COLUMN}")
    for i, row in frame.iterrows():
        try:
            chunk = get_student_input(row)
            frame.at[i, "student_input"] = chunk
            if require_targets:
                if row.get("generation_status", "completed") != "completed":
                    raise ValueError("Test references need generation_status='completed'; fix failed teacher rows first")
                if row.get("summary_type", RUN_SUMMARY_TYPE) != RUN_SUMMARY_TYPE:
                    raise ValueError("Teacher summary_type differs from this run")
                frame.at[i, "target_text"] = validate_reference(row[TARGET_COLUMN], chunk, row)
        except Exception as exc:
            raise ValueError(f"{path.name}, segment {row['segment_id']}: {exc}") from exc
    return frame


def event_ids(frame):
    return {eid for text in frame.student_input for eid in ID_TAG.findall(text)}


## 4. Fixed, five-event demonstration bank for few-shot



In [ ]:
# ============================================================================
# FIXED FEW-SHOT EXAMPLE BANK — EDIT THIS SECTION IF FEW-SHOT Needed
# ============================================================================
EXAMPLE_BANK = {
 'linux': {'extractive': None, 'abstractive': None, 'hybrid': None},
 'Apache': {'extractive': None, 'abstractive': None, 'hybrid': None},
 'zookeeper': {'extractive': None, 'abstractive': None, 'hybrid': None}}


In [ ]:
# Test-only inference using the explicitly configured TEST_FILE.
if not EVAL_FILE.exists(): raise FileNotFoundError(EVAL_FILE)
eval_frame = load_frame(EVAL_FILE, INFERENCE_SPLIT, require_targets=True)
if MAX_SEGMENTS is not None: eval_frame = eval_frame.iloc[:MAX_SEGMENTS].copy()

# Export naming only: dataset, window size, summary type and inference mode.
# Prefer the wN token in the input filename; otherwise infer the segment size
# from the original (untruncated) first test segment.
def resolve_export_window_size():
    match = re.search(r'(?:^|[_-])w(\d+)(?=[_.-]|$)', EVAL_FILE.name, re.I)
    if match:
        return int(match.group(1))
    if eval_frame.empty:
        raise ValueError('Cannot infer window size from an empty test file without wN in its name')
    if 'events_json' in eval_frame.columns:
        events = eval_frame.iloc[0]['events_json']
        if isinstance(events, str) and events.strip():
            parsed = json.loads(events)
            if isinstance(parsed, list) and parsed:
                return len(parsed)
    lines = [line for line in str(eval_frame.iloc[0]['student_input']).splitlines() if line.strip()]
    if not lines:
        raise ValueError('Cannot infer window size from the test data')
    return len(lines)

WINDOW_SIZE = resolve_export_window_size()
EXPORT_PREFIX = f'{SYSTEM_KEY}_w{WINDOW_SIZE}_{SUMMARY_TYPE}'
def export_mode_prefix(mode):
    return f'{EXPORT_PREFIX}_{mode}'
def export_run_dir(mode):
    suffix = f'{mode}_k{FEW_SHOT_K}' if mode == 'few_shot' else mode
    return OUTPUT_ROOT / SYSTEM_KEY / APPROACH / SUMMARY_TYPE / f'{EXPORT_PREFIX}_{suffix}_json_v4'
print('Output filename prefix:', EXPORT_PREFIX)


def get_fixed_demonstrations(system_key, summary_type):
    slot = EXAMPLE_BANK[system_key][summary_type]
    if slot is None:
        raise ValueError(f"No examples for {system_key}/{summary_type}. Fill EXAMPLE_BANK or set MODES=['zero_shot'].")
    if not isinstance(slot,dict) or set(slot)!={'normal','fault'}:
        raise ValueError('Each populated slot needs exactly normal and fault entries')
    rows=[]
    for role in ('normal','fault'):
        ex=slot[role]
        for k in ('source_segment_id','source_event_ids','student_input','target_text'):
            if k not in ex: raise ValueError(f'Missing {role}.{k}')
        if len(ex['source_event_ids']) != 5: raise ValueError(f'{role}: require five event IDs')
        text=ex['student_input']
        if LABEL_TAG.search(text) or re.search(r'\bEventTemplate\s*=',text,re.I):
            raise ValueError(f'{role}: leaked annotation or EventTemplate')
        ids=ID_TAG.findall(text)
        if len(text.splitlines())!=5 or ids!=[str(x) for x in ex['source_event_ids']] or len(set(ids))!=5:
            raise ValueError(f'{role}: must have five unique, matching original ID-bearing event lines')
        # Strictly validate against original teacher schema, exact source copying, and normal phrase.
        normalized=validate_target(ex['target_text'],text)
        is_fault=has_fault_output(normalized)
        if is_fault != (role=='fault'):
            raise ValueError(f'{role}: target fault presence differs from its example role')
        rows.append({'segment_id':ex['source_segment_id'], 'student_input':text,
                     'target_text':normalized,'teacher_fault':is_fault,
                     'source_event_ids':ids,'example_role':role})
    if rows[0]['segment_id']==rows[1]['segment_id']:
        raise ValueError('Normal and fault demonstrations must use different training segments')
    if set(rows[0]['source_event_ids']) & set(rows[1]['source_event_ids']):
        raise ValueError('Five-event demonstrations overlap')
    return pd.DataFrame(rows)

if 'few_shot' in MODES:
    demonstrations=get_fixed_demonstrations(SYSTEM_KEY,SUMMARY_TYPE)
    demonstration_ids=demonstrations.segment_id.tolist()
    # Demonstrations are fixed in EXAMPLE_BANK; validate their format and targets
    # above, without requiring a separate training CSV during test inference.
    # No overlap with ALL evaluation rows, not only first MAX_SEGMENTS.
    complete_eval_ids=set()
    full_eval_raw=pd.read_csv(EVAL_FILE,dtype=str,keep_default_na=False,encoding='utf-8-sig')
    for ejson in full_eval_raw.events_json:
        for i,ev in enumerate(json.loads(ejson)):
            eid=event_identifier(ev,i)
            if eid:complete_eval_ids.add(eid)
    overlap=set(e for xs in demonstrations.source_event_ids for e in xs)&complete_eval_ids
    if overlap:raise ValueError(f'Few-shot examples overlap with {INFERENCE_SPLIT}: {sorted(overlap)[:5]}')
else:
    demonstrations=pd.DataFrame(columns=['segment_id','student_input','target_text','teacher_fault'])
    demonstration_ids=[]
print('Evaluation split:',INFERENCE_SPLIT,'segments:',len(eval_frame))
print('Selected fixed demonstrations:',demonstration_ids)


## 5. Load original, unmodified base model (no LoRA adapter)

In [ ]:
# Import Unsloth before transformers and PEFT.
from unsloth import FastLanguageModel
import torch
from transformers import set_seed
if not torch.cuda.is_available(): raise RuntimeError('GPU runtime required')
set_seed(SEED)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
)
FastLanguageModel.for_inference(model)
model.eval()
model.config.use_cache=True
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
print('Loaded base model without LoRA:',MODEL_NAME)
print('GPU:',torch.cuda.get_device_name(0))


## 6. Construct zero-shot and fixed few-shot prompts; verify no label leakage

In [ ]:
def build_baseline_prompt(chunk, mode):
    # This is EXACTLY the original V2.4 prompt for zero shot.
    base_prompt = build_prompt(chunk)
    if mode == 'zero_shot': return base_prompt
    if mode != 'few_shot': raise ValueError(mode)
    # Fixed, five-event demonstrations come only from teacher-labelled training segments.
    demonstrations_text = ''.join(
        '\n\nLog chunk:\n' + row.student_input + '\n\n### Response:\n' + row.target_text
        for _,row in demonstrations.iterrows())
    return RUN_INSTRUCTION + demonstrations_text + '\n\nLog chunk:\n' + chunk + '\n\n### Response:\n'

def prompt_safety_check(prompt,chunk,mode):
    if LABEL_TAG.search(prompt) or re.search(r'\bEventTemplate\s*=',prompt,re.I):
        raise ValueError('Leaked GT event label or EventTemplate into prompt')
    if mode=='zero_shot' and prompt != build_prompt(chunk):
        raise ValueError('Zero-shot prompt must match V2.4 byte-for-byte')
    if mode=='few_shot' and not all(x in prompt for x in demonstrations.target_text.tolist()):
        raise ValueError('Few-shot examples missing')

for mode in MODES:
    for chunk in eval_frame.student_input.head(min(5,len(eval_frame))):
        prompt_safety_check(build_baseline_prompt(chunk,mode),chunk,mode)
    print(mode,'example prompt tokens:',len(tokenizer(build_baseline_prompt(eval_frame.student_input.iloc[0],mode),add_special_tokens=True)['input_ids']))


zero_shot example prompt tokens: 2022


## 7. Inference with synchronized CUDA timings, exact token counts and resume protection

In [ ]:
def extract_summary_json(raw, chunk):
    """Find first complete JSON object matching the required summary schema.

    Preserve original text separately. Ignore braces within quoted JSON strings.
    Never fabricate, repair or merge generated JSON objects.
    """
    decoder = json.JSONDecoder()
    required = ({"explanation", "summary"} if RUN_SUMMARY_TYPE == 'abstractive'
                else {"explanation", "critical_events", "context_events"})
    if RUN_SUMMARY_TYPE == 'hybrid':
        required |= {"analysis", "interpretation"}
    first_schema_json = ''
    first_validation_error = ''
    for match in re.finditer(r'\{', raw):
        try:
            obj, end = decoder.raw_decode(raw, match.start())
        except (json.JSONDecodeError, ValueError):
            continue
        if not isinstance(obj, dict) or set(obj) != required:
            continue
        candidate = raw[match.start():end]
        if not first_schema_json:
            first_schema_json = candidate
        try:
            validated = validate_target(candidate, chunk)
            return candidate, validated, '', True
        except Exception as exc:
            if not first_validation_error:
                first_validation_error = f'{type(exc).__name__}: {exc}'
    if first_schema_json:
        return first_schema_json, '', first_validation_error, True
    return '', '', 'No complete schema-matching JSON object found', False

def synchronize():
    if torch.cuda.is_available(): torch.cuda.synchronize()

def generate_one(chunk,mode):
    raw=clean=error=extracted_json=''; status='failed'; json_structurally_valid=False
    usage={
        'student_input_tokens':None,'student_output_tokens':0,'student_total_tokens':None,
        'student_generation_time_seconds':None,'student_end_to_end_time_seconds':None,
        'student_tokens_per_second':None,'student_peak_cuda_memory_bytes':None,
        'student_eos_reached':False,'student_prompt_tokens_without_demos':None,
        'student_demonstration_tokens':0,
    }
    t_start=time.perf_counter()
    try:
        prompt=build_baseline_prompt(chunk,mode)
        prompt_safety_check(prompt,chunk,mode)
        inputs=tokenizer(prompt,return_tensors='pt',truncation=False).to(next(model.parameters()).device)
        count=int(inputs['input_ids'].shape[1]); usage['student_input_tokens']=count
        base_count=len(tokenizer(build_prompt(chunk),truncation=False)['input_ids'])
        usage['student_prompt_tokens_without_demos']=base_count
        usage['student_demonstration_tokens']=count-base_count
        if count + MAX_NEW_TOKENS > MAX_SEQ_LENGTH:
            raise ValueError(f'Prompt {count} + output budget {MAX_NEW_TOKENS} exceeds context {MAX_SEQ_LENGTH}. No truncation.')
        synchronize()
        if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        t_gen=time.perf_counter()
        try:
            with torch.inference_mode():
                result=model.generate(**inputs,max_new_tokens=MAX_NEW_TOKENS,max_length=None,do_sample=False,
                                      use_cache=True,eos_token_id=tokenizer.eos_token_id,
                                      pad_token_id=tokenizer.pad_token_id)
        finally:
            synchronize()
            usage['student_generation_time_seconds']=time.perf_counter()-t_gen
            if torch.cuda.is_available(): usage['student_peak_cuda_memory_bytes']=int(torch.cuda.max_memory_allocated())
        ids=result[0,count:]
        usage['student_output_tokens']=int(ids.numel())
        usage['student_eos_reached']=bool(len(ids) and ids[-1].item()==tokenizer.eos_token_id)
        raw=tokenizer.decode(ids,skip_special_tokens=True)
        if not raw.strip():
            error='Empty decoded response'
        else:
            extracted_json, clean, error, json_structurally_valid = extract_summary_json(raw, chunk)
            if clean:
                status='completed'
            elif not usage['student_eos_reached']:
                error += ' | Generation did not reach EOS (output may be truncated)'
    except Exception as e:
        error=f'{type(e).__name__}: {e}'
        if isinstance(e,RuntimeError) and ('CUDA' in str(e) or 'out of memory' in str(e).lower()):
            raise
    finally:
        usage['student_end_to_end_time_seconds']=time.perf_counter()-t_start
        if usage['student_input_tokens'] is not None:
            usage['student_total_tokens']=usage['student_input_tokens']+usage['student_output_tokens']
        sec=usage['student_generation_time_seconds']
        usage['student_tokens_per_second']=usage['student_output_tokens']/sec if sec and sec>0 else None
    return raw,extracted_json,clean,status,error,json_structurally_valid,usage

METRIC_COLS=['student_input_tokens','student_output_tokens','student_total_tokens',
             'student_generation_time_seconds','student_end_to_end_time_seconds',
             'student_tokens_per_second','student_peak_cuda_memory_bytes',
             'student_eos_reached','student_prompt_tokens_without_demos',
             'student_demonstration_tokens']

def run_inference(mode):
    suffix=f'{mode}_k{FEW_SHOT_K}' if mode=='few_shot' else 'zero_shot'
    run_dir=export_run_dir(mode)
    run_dir.mkdir(parents=True,exist_ok=True)
    RUN_SIGNATURE={
        'version':'logsumm-baselines-v5-parity-token-metrics', 'mode':mode,'system_key':SYSTEM_KEY,
        'approach':APPROACH,'summary_type':SUMMARY_TYPE,'model_name':MODEL_NAME,
        'load_in_4bit':LOAD_IN_4BIT,'max_seq_length':MAX_SEQ_LENGTH,'max_new_tokens':MAX_NEW_TOKENS,
        'seed':SEED,'prompt_instruction_sha256':digest(RUN_INSTRUCTION),
        'prompt_format':'plain-text-v3-compact','decoding':{'do_sample':False},
        'few_shot_k':FEW_SHOT_K if mode=='few_shot' else 0,
        'demonstration_ids':demonstration_ids if mode=='few_shot' else [],
        'demonstration_event_ids':[list(x) for x in demonstrations.source_event_ids] if mode=='few_shot' else [],
        'demonstration_summary_hashes':[digest(x) for x in demonstrations.target_text.tolist()] if mode=='few_shot' else [],
        'evaluation_file':str(EVAL_FILE),'evaluation_sha256':file_hash(EVAL_FILE),
        'example_bank_sha256':digest(EXAMPLE_BANK[SYSTEM_KEY][SUMMARY_TYPE]) if mode=='few_shot' else None,
    }
    signature=digest(RUN_SIGNATURE)
    save_json({**RUN_SIGNATURE,'config_hash':signature,
               'gpu':torch.cuda.get_device_name(0),
               'torch_version':torch.__version__,
               'transformers_version':importlib.metadata.version('transformers'),
               'unsloth_version':importlib.metadata.version('unsloth')},run_dir/f'{export_mode_prefix(mode)}_run_config.json')
    out=f'student_{RUN_SUMMARY_TYPE}'
    predictions_path=run_dir/f'{export_mode_prefix(mode)}_{INFERENCE_SPLIT}_predictions.csv'
    checkpoint_path=run_dir/f'{export_mode_prefix(mode)}_{INFERENCE_SPLIT}_checkpoint.csv'
    # checkpoint_path=Path("------")
    new=eval_frame.copy().reset_index(drop=True)
    cols=[out,out+'_raw',out+'_json_extracted',out+'_json_structurally_valid',out+'_status',out+'_error', 'inference_config', 'inference_mode',
          'few_shot_segment_ids', 'base_model_id', *METRIC_COLS]
    # Do not save internal normalized target text as an additional data column.
    new=new.drop(columns=cols+['target_text'],errors='ignore')
    results=new.copy()
    # Object dtype accepts numeric token counts, timings, booleans, and strings.
    for c in cols:
        results[c] = pd.Series([''] * len(results), index=results.index, dtype=object)
    results[out+'_status']='pending'
    results['inference_config']=signature
    results['inference_mode']=mode
    results['few_shot_segment_ids']=json.dumps(demonstration_ids if mode=='few_shot' else [])
    results['base_model_id']=MODEL_NAME
    # Resume only from the matching mode/split checkpoint (or completed predictions).
    # Report exactly what was recovered; never silently start from zero.
    resume_source = checkpoint_path if checkpoint_path.exists() else (predictions_path if predictions_path.exists() else None)
    if INFERENCE_RESUME and resume_source is not None:
        old=pd.read_csv(resume_source,dtype=str,keep_default_na=False,encoding='utf-8-sig')
        if (set(results.columns)!=set(old.columns) or not old['inference_config'].eq(signature).all()
            or not old[new.columns].equals(new)):
            raise ValueError('Resume data/config mismatch; choose a new run directory or disable resume')
        results=old[results.columns].copy().astype(object)
        print(f'{mode}: RESUMING from {resume_source}',flush=True)
    elif INFERENCE_RESUME:
        raise FileNotFoundError(
            f'Resume requested, but neither checkpoint nor predictions file was found. '
            f'Expected checkpoint: {checkpoint_path}. Expected predictions: {predictions_path}. '
            'Restore the correct checkpoint or explicitly set INFERENCE_RESUME=False for a new run.'
        )
    else:
        print(f'{mode}: INFERENCE_RESUME=False; starting a new run.',flush=True)

    # Resume policy: completed valid-output rows are retained; optionally retain
    # previously attempted failed rows too, so only never-attempted rows are run.
    # A failed row must have an actual saved attempt (raw response or token usage),
    # otherwise it is eligible for generation.
    statuses=results[out+'_status'].astype(str).str.strip().str.lower()
    nonempty=results[out].astype(str).str.strip().ne('')
    attempted=(results[out+'_raw'].astype(str).str.strip().ne('') |
               results['student_output_tokens'].astype(str).str.strip().ne(''))
    completed_skip=statuses.eq('completed') & nonempty
    failed_skip=statuses.eq('failed') & attempted & (not RESUME_RETRY_FAILED)
    resume_skip=(completed_skip | failed_skip).to_numpy()
    print(f'{mode}: total={len(results)}, completed skipped={int(completed_skip.sum())}, '
          f'previously failed skipped={int(failed_skip.sum())}, '
          f'pending/remaining to generate={int((~resume_skip).sum())}; '
          f'failed retry enabled={RESUME_RETRY_FAILED}',flush=True)
    processed_since_resume=0
    try:
        for i,row in results.iterrows():
            if resume_skip[i]:
                continue
            raw,extracted_json,clean,status,error,json_structurally_valid,usage=generate_one(row.student_input,mode)
            results.at[i,out]=clean
            results.at[i,out+'_raw']=raw
            results.at[i,out+'_json_extracted']=extracted_json
            results.at[i,out+'_json_structurally_valid']=json_structurally_valid
            results.at[i,out+'_status']=status
            results.at[i,out+'_error']=error
            for key,value in usage.items():results.at[i,key]=value
            if PRINT_EACH_RESPONSE:
                print("\n" + "=" * 72)
                print(f"Mode: {mode} | Test sample: {i+1}/{len(results)} | Segment: {row.get('segment_id', 'N/A')}")
                print(f"Status: {status} | Input tokens: {usage['student_input_tokens']} | Generated tokens: {usage['student_output_tokens']} | Generation seconds: {usage['student_generation_time_seconds']}")
                print("RAW MODEL RESPONSE:")
                shown_raw = raw if PRINT_MAX_CHARS is None else raw[:PRINT_MAX_CHARS]
                print(shown_raw if shown_raw else '[EMPTY]')
                if PRINT_MAX_CHARS is not None and len(raw) > PRINT_MAX_CHARS:
                    print(f"[Console preview truncated: {len(raw)} total characters; full response saved to CSV]")
                print("EXTRACTED JSON:")
                print(extracted_json if extracted_json else '[NO COMPLETE SCHEMA-MATCHING JSON]')
                print("STRUCTURALLY MATCHES JSON SCHEMA:", json_structurally_valid)
                print("VALIDATED SUMMARY:")
                print(clean if clean else '[EMPTY OR INVALID]')
                if error:
                    print("ERROR:", error)
                print("=" * 72, flush=True)
            processed_since_resume += 1
            if processed_since_resume % CHECKPOINT_EVERY == 0:
                save_csv(results,checkpoint_path)
                print(f'{mode}: checkpoint saved; newly processed={processed_since_resume}; latest row={i+1}/{len(results)}; latest status={status}', flush=True)
    finally:
        save_csv(results,checkpoint_path)
    save_csv(results,predictions_path)
    return results,run_dir


## 8. Teacher-agreement and efficiency evaluation (separate from independent anomaly ground truth)

In [ ]:
def inference_usage_summary(predictions,status_col):
    attempted=pd.to_numeric(predictions['student_end_to_end_time_seconds'],errors='coerce').notna()
    data=predictions.loc[attempted].copy()
    def numeric(name):return pd.to_numeric(data[name],errors='coerce')
    def total(name):return float(numeric(name).sum())
    duration=total('student_generation_time_seconds')
    output=total('student_output_tokens')
    valid=predictions[status_col].eq('completed')
    return {
        'segments':len(predictions), 'attempted_segments':int(attempted.sum()),
        'valid_segments':int(valid.sum()),'valid_rate':float(valid.mean()),
        'input_tokens_total':int(total('student_input_tokens')),
        'generated_tokens_total':int(output),
        'total_tokens':int(total('student_total_tokens')),
        'demo_tokens_total':int(total('student_demonstration_tokens')),
        'generation_seconds_total':duration,
        'end_to_end_seconds_total':total('student_end_to_end_time_seconds'),
        'mean_generation_seconds_per_attempt':float(numeric('student_generation_time_seconds').mean()),
        'median_generation_seconds_per_attempt':float(numeric('student_generation_time_seconds').median()),
        'aggregate_generated_tokens_per_second':output/duration if duration>0 else None,
        'mean_peak_cuda_memory_bytes':float(numeric('student_peak_cuda_memory_bytes').mean()),
    }


In [ ]:
def evaluate_predictions(predictions, split):
    """Diagnostics against teacher summaries, not a separate anomaly detector.
    Failed outputs remain in coverage denominators, rather than being dropped.
    Event agreement uses message text so choosing a different occurrence of a
    repeated message does not count as an error.
    """
    out = f"student_{RUN_SUMMARY_TYPE}"
    valid = predictions[out + "_status"].eq("completed")
    metrics = {"split": split, "chunks": len(predictions),
               "valid_outputs": int(valid.sum()), "valid_output_rate": float(valid.mean())}
    matched = 0
    full_matched = 0
    explanation_matched = 0
    tp = fp = fn = 0
    chunk_tp = chunk_fp = chunk_fn = chunk_tn = 0
    failed_with_critical = 0
    for i, row in predictions.iterrows():
        if row.get("generation_status", "completed") != "completed" or row.get("summary_type", RUN_SUMMARY_TYPE) != RUN_SUMMARY_TYPE:
            raise ValueError("Metrics require completed teacher summaries of the same type")
        reference = validate_reference(row[TARGET_COLUMN], row.student_input, row)
        predicted = row[out] if valid.loc[i] else None
        if predicted is not None:
            predicted = validate_target(predicted, row.student_input)
            matched += int(summary_payload(predicted) == summary_payload(reference))
            full_matched += int(json.loads(predicted) == json.loads(reference))
            explanation_matched += int(json.loads(predicted)["explanation"] == json.loads(reference)["explanation"])
        if RUN_SUMMARY_TYPE == "abstractive":
            actual = has_fault_output(reference)
            if predicted is None:
                failed_with_critical += int(actual)
            else:
                present = has_fault_output(predicted)
                chunk_tp += int(actual and present); chunk_fp += int(not actual and present)
                chunk_fn += int(actual and not present); chunk_tn += int(not actual and not present)
            continue
        message_by_id = {event_identifier(e, j): e.get("Content", "") for j, e in enumerate(json.loads(row["events_json"]))} if row.get("events_json", "") else {}
        def keys(summary):
            return {message_by_id.get(ID_TAG.findall(line)[0], whitespace_key(line))
                    for line in json.loads(summary)["critical_events"]}
        gold = keys(reference)
        if predicted is None:
            fn += len(gold)
            failed_with_critical += int(bool(gold))
            continue
        pred = keys(predicted)
        tp += len(gold & pred); fp += len(pred - gold); fn += len(gold - pred)
        chunk_tp += int(bool(gold) and bool(pred)); chunk_fp += int(not gold and bool(pred))
        chunk_fn += int(bool(gold) and not pred); chunk_tn += int(not gold and not pred)
    metrics["teacher_exact_match_rate"] = matched / len(predictions)  # Summary payload only.
    metrics["summary_only_exact_match_rate"] = matched / len(predictions)
    metrics["full_response_exact_match_rate"] = full_matched / len(predictions)
    metrics["explanation_exact_match_rate"] = explanation_matched / len(predictions)  # Descriptive, not explanation quality.
    if RUN_SUMMARY_TYPE != "abstractive":
        metrics.update(
            critical_message_precision=tp / (tp + fp) if tp + fp else None,
            critical_message_recall=tp / (tp + fn) if tp + fn else None,
            critical_message_f1=2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else None,
            critical_tp=tp, critical_fp=fp, critical_fn=fn,
            valid_chunk_critical_presence_tp=chunk_tp, valid_chunk_critical_presence_fp=chunk_fp,
            valid_chunk_critical_presence_fn=chunk_fn, valid_chunk_critical_presence_tn=chunk_tn,
            failed_chunks_with_teacher_critical_events=failed_with_critical,
        )
    if RUN_SUMMARY_TYPE == "abstractive":
        denom = 2 * chunk_tp + chunk_fp + chunk_fn
        metrics.update(
            valid_prefix_chunk_tp=chunk_tp, valid_prefix_chunk_fp=chunk_fp,
            valid_prefix_chunk_fn=chunk_fn, valid_prefix_chunk_tn=chunk_tn,
            valid_prefix_chunk_precision=chunk_tp / (chunk_tp + chunk_fp) if chunk_tp + chunk_fp else None,
            valid_prefix_chunk_recall=chunk_tp / (chunk_tp + chunk_fn) if chunk_tp + chunk_fn else None,
            valid_prefix_chunk_f1=2 * chunk_tp / denom if denom else None,
            valid_prefix_chunk_accuracy=(chunk_tp + chunk_tn) / int(valid.sum()) if valid.sum() else None,
            failed_anomalous_summaries=failed_with_critical,
        )
    metrics["student_inference_efficiency"] = inference_usage_summary(predictions, out + "_status")
    save_json(metrics, RUN_DIR / f"{split}_summary_metrics.json")
    print(json.dumps(metrics, indent=2))
    if RUN_SUMMARY_TYPE == "abstractive":
        print("Exact match is descriptive only; paraphrases need factual/semantic review and downstream evaluation.")
    return metrics


## 9. Execute selected modes and export comparable metrics

In [ ]:
completed_prediction_files = []
for mode in MODES:
    prediction_frame,run_dir=run_inference(mode)
    RUN_DIR=run_dir  # used only for writing split-level metrics in the V2.4 evaluator
    metrics=evaluate_predictions(prediction_frame,INFERENCE_SPLIT)
    save_json(metrics,run_dir/f'{export_mode_prefix(mode)}_{INFERENCE_SPLIT}_evaluation.json')
    print('Saved',run_dir)
    completed_prediction_files.append(run_dir / f'{export_mode_prefix(mode)}_{INFERENCE_SPLIT}_predictions.csv')

# Automatic downloads start only after every selected mode finishes.
auto_download_prediction_csvs(completed_prediction_files)


## Download a ZIP containing all generated inference files
Run the next cell whenever you want an archive of currently generated files, including partial checkpoints.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import zipfile

ZIP_SOURCE_DIRS = [export_run_dir(mode) for mode in MODES]
ZIP_MODE_TAG = "_".join(MODES)
ZIP_DESTINATION = (OUTPUT_ROOT / SYSTEM_KEY / APPROACH / SUMMARY_TYPE /
                   f"{EXPORT_PREFIX}_{ZIP_MODE_TAG}_inference_files.zip")
ZIP_DESTINATION.parent.mkdir(parents=True, exist_ok=True)
number_of_files = 0
with zipfile.ZipFile(ZIP_DESTINATION, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for source_dir in ZIP_SOURCE_DIRS:
        if not source_dir.exists():
            print("No output directory yet:", source_dir)
            continue
        for file_path in sorted(source_dir.rglob("*")):
            if file_path.is_file() and file_path.resolve() != ZIP_DESTINATION.resolve():
                archive.write(file_path, arcname=str(Path(source_dir.name) / file_path.relative_to(source_dir)))
                number_of_files += 1
print(f"ZIP created: {ZIP_DESTINATION} | Files: {number_of_files} | Bytes: {ZIP_DESTINATION.stat().st_size}")
if number_of_files:
    try:
        from google.colab import files as colab_files
        colab_files.download(str(ZIP_DESTINATION))
    except ImportError:
        print("Download manually from:", ZIP_DESTINATION)


ZIP created: /content/logsumm_baselines/zookeeper/stratified/abstractive/zookeeper_w5_abstractive_zero_shot_inference_files.zip | Files: 5 | Bytes: 652575


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>